In [ ]:
# CELL 1: Setup and Kaggle Download
import os
from google.colab import files

# 1. Prompt you to upload the kaggle.json file you downloaded earlier
print("Upload your kaggle.json file:")
files.upload()

# 2. Move the file to the correct hidden directory and set permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Download the dataset directly from Kaggle (Fruits 360)
print("Downloading dataset from Kaggle...")
!kaggle datasets download -d moltean/fruits

# 4. Unzip the downloaded file silently into the Colab environment
print("Unzipping dataset...")
!unzip -q fruits.zip -d /content/dataset
print("Data ready at /content/dataset!")

In [ ]:
# CELL 2: Data Loading and Preprocessing
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Define paths (Fruits 360 extracts into these specific subfolders)
base_dir = '/content/dataset/fruits-360_100x100/fruits-360'
train_dir = os.path.join(base_dir, 'Training')
test_dir = os.path.join(base_dir, 'Test')

# Image parameters
IMG_SHAPE = (100, 100) # Fruits 360 images are 100x100
BATCH_SIZE = 32

# Augmentation for training data
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2 # Use 20% of training data for validation
)

# Only rescale for testing/validation
test_datagen = ImageDataGenerator(rescale=1./255)

print("Loading Training Data...")
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

print("Loading Validation Data...")
val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# Save the class names mapped to their indices (crucial for the FastAPI backend later)
class_indices = train_generator.class_indices
class_names = {v: k for k, v in class_indices.items()}
print(f"Total Classes: {len(class_names)}")

In [ ]:
# CELL 3: Building the CNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Number of fruit classes detected dynamically
NUM_CLASSES = train_generator.num_classes

model = Sequential([
    # First Convolutional Block
    Conv2D(32, (3, 3), activation='relu', input_shape=(100, 100, 3)),
    MaxPooling2D(2, 2),
    
    # Second Convolutional Block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    
    # Third Convolutional Block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    
    # Flatten and Dense Layers
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5), # Prevent overfitting
    Dense(NUM_CLASSES, activation='softmax') # Output layer
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# CELL 4: Training the Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Ensure we save only the weights with the highest validation accuracy
checkpoint = ModelCheckpoint(
    'best_fruit_model.h5', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max',
    verbose=1
)

# Stop training if accuracy stops improving to save time
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

EPOCHS = 15

print("Starting training on T4 GPU...")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint, early_stop]
)

In [ ]:
# CELL 5: Export for Backend Deployment
import json
from google.colab import files

# Save the final model explicitly 
model.save('final_fruit_model.h5')

# Save the class dictionary to a JSON file so your FastAPI backend knows what ID 0, 1, 2 actually mean
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)

print("Exporting files to local machine...")
files.download('final_fruit_model.h5')
files.download('class_names.json')